In [5]:
from pathlib import Path
import re
from collections import defaultdict
from collections import Counter, defaultdict
from bs4 import BeautifulSoup

In [ ]:
ROOT = Path("data/annotated")
SPLITS = ["train", "test", "dev"]

# Parent labels
PARENT_LABELS = [
    "decision",
    "legislation",
    "secondary sources",
    "unable to classify"
]

# Child/sub labels
SUBLABELS = [
    "authors",
    "title",
    "fragment",
    "citation",
    "source",
]

In [ ]:


per_split_totals = defaultdict(int)

for split in SPLITS:
    split_dir = ROOT / split

    if not split_dir.exists():
        print(f"skip (not found): {split_dir}")
        continue

    files = sorted(split_dir.rglob("*.html"))

    if not files:
        print(f"no html files in: {split_dir}")
        continue

    print(f"--- split: {split} (files: {len(files)}) ---")

    for f in files:
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")

            soup = BeautifulSoup(text, "html.parser")

            # Find all manual_label and auto_label tags
            tags = soup.find_all(["manual_label", "auto_label"])

            count = len(tags)

            per_split_totals[split] += count

            rel = f.relative_to(ROOT)
            print(f"{rel}: {count}")

        except Exception as e:
            print(f"error processing {f}: {e}")

    print()

print("\n--- per-split totals ---")

for split in SPLITS:
    if split in per_split_totals:
        print(f"{split}: {per_split_totals[split]}")

--- split: train (files: 5) ---
train\1994CanLII4528NLCA.html: 424
train\1997CanLII16226ONCA.html: 1911
train\2008CSC9.html: 1399
train\2016NBOMB12.html: 306
train\2019SCC65.html: 4658

--- split: test (files: 3) ---
test\1989CanLII1415ONCA.html: 241
test\2005QCCA437.html: 312
test\2024NBKB203.html: 658

--- split: dev (files: 3) ---
dev\2001CanLII21117.html: 789
dev\2002SCC33.html: 953
dev\2021QCCA1675.html: 272


--- per-split totals ---
train: 8698
test: 1211
dev: 2014


In [ ]:


counts = Counter()

for split in SPLITS:
    split_dir = ROOT / split

    html_files = list(split_dir.rglob("*.html"))

    for html_file in html_files:
        try:
            with open(html_file, "r", encoding="utf-8") as f:
                soup = BeautifulSoup(f, "html.parser")

            # Find all manual_label and auto_label tags
            tags = soup.find_all(["manual_label", "auto_label"])

            for tag in tags:
                label = tag.get("labelname")

                if label:
                    counts[label] += 1

        except Exception as e:
            print(f"Error processing {html_file}: {e}")

# =========================
# PRINT FINAL TABLE
# =========================

print("\n" + "=" * 80)
print("TOTAL LABEL COUNTS")
print("=" * 80)

print(f"{'LABEL':<20} {'COUNT':>10}")
print("-" * 32)

# Parent labels
parent_total = 0
for label in PARENT_LABELS:
    c = counts[label]
    parent_total += c
    print(f"{label:<20} {c:>10}")

print("-" * 32)
print(f"{'TOTAL PARENT':<20} {parent_total:>10}")

print()

# Sublabels
sub_total = 0
for label in SUBLABELS:
    c = counts[label]
    sub_total += c
    print(f"{label:<20} {c:>10}")

print("-" * 32)
print(f"{'TOTAL SUBLABELS':<20} {sub_total:>10}")

print()

# Grand total
grand_total = parent_total + sub_total
print(f"{'TOTAL COUNT':<20} {grand_total:>10}")


TOTAL LABEL COUNTS
LABEL                     COUNT
--------------------------------
decision                   2177
legislation                1533
secondary sources           348
unable to classify            4
--------------------------------
TOTAL PARENT               4062

authors                     280
title                      2992
fragment                   2558
citation                   1816
source                      215
--------------------------------
TOTAL SUBLABELS            7861

TOTAL COUNT               11923
